# 🤖 Model Training — Peddapalli Accident Risk Prediction

Trains and evaluates a **VotingClassifier** ensemble (RandomForest + GradientBoosting) to predict `accident_occurred`.

**Pipeline:**
1. Load & encode features
2. Train/test split (80/20)
3. Hyperparameter tuning with GridSearchCV
4. Final model evaluation
5. Save artifacts (`model.pkl`, `scaler.pkl`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, os
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             accuracy_score, f1_score)
from sklearn.pipeline import Pipeline

print("All imports loaded ✅")

## 1. Load & Encode

In [ ]:
df = pd.read_csv('../backend/data/peddapalli_accidents.csv')

ROAD_TYPE_MAP = {
    'Highway':0,'Bypass':1,'District Road':2,'Bridge Road':3,
    'State Highway':4,'Industrial Road':5,'Village Road':6,'Urban Road':7
}
WEATHER_MAP    = {'Clear':0,'Rainy':1,'Foggy':2,'Overcast':3,'Night Drizzle':4}
ROAD_COND_MAP  = {'Good':0,'Potholed':1,'Wet':2,'Under Construction':3,'Gravelled':4}
LIGHT_MAP      = {'Daylight':0,'Dark – No Street Light':1,'Dark – Street Light':2,'Dawn/Dusk':3}
VEHICLE_MAP    = {'Two-Wheeler':0,'Car/Jeep':1,'Auto-Rickshaw':2,'Truck/Lorry':3,'Bus':4,'Tractor':5}
COLLISION_MAP  = {'Head-On':0,'Rear-End':1,'Side-Swipe':2,'Rollover':3,'Hit-and-Run':4,'Pedestrian':5}

df['road_type_enc']       = df['road_type'].map(ROAD_TYPE_MAP)
df['weather_enc']         = df['weather'].map(WEATHER_MAP)
df['road_condition_enc']  = df['road_condition'].map(ROAD_COND_MAP)
df['light_condition_enc'] = df['light_condition'].map(LIGHT_MAP)
df['vehicle_type_enc']    = df['vehicle_type'].map(VEHICLE_MAP)
df['collision_type_enc']  = df['collision_type'].map(COLLISION_MAP)

FEATURE_COLS = [
    'hour','is_weekend','is_peak_hour','is_night',
    'speed_limit_kmph','traffic_volume','vehicles_involved',
    'road_type_enc','weather_enc','road_condition_enc',
    'light_condition_enc','vehicle_type_enc','collision_type_enc'
]

X = df[FEATURE_COLS].values
y = df['accident_occurred'].values
print(f"X shape: {X.shape}")
print(f"y balance: {np.bincount(y)}")

## 2. Train/Test Split + Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")

## 3. Baseline Models

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

baselines = {
    'Dummy (majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'RandomForest (default)': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting (default)': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':<30} {'Accuracy':>10} {'F1':>10}")
print('-' * 52)
for name, clf in baselines.items():
    clf.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_s))
    f1  = f1_score(y_test, clf.predict(X_test_s))
    print(f"{name:<30} {acc:>10.4f} {f1:>10.4f}")

## 4. Hyperparameter Tuning (GridSearchCV)

In [ ]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5],
}

gb_params = {
    'n_estimators':    [100, 200],
    'learning_rate':   [0.05, 0.1],
    'max_depth':       [3, 5],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Tuning RandomForest...")
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                       rf_params, cv=skf, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train_s, y_train)
print(f"  Best RF params: {rf_grid.best_params_}")
print(f"  Best RF CV F1:  {rf_grid.best_score_:.4f}")

print("Tuning GradientBoosting...")
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
                       gb_params, cv=skf, scoring='f1', n_jobs=-1)
gb_grid.fit(X_train_s, y_train)
print(f"  Best GB params: {gb_grid.best_params_}")
print(f"  Best GB CV F1:  {gb_grid.best_score_:.4f}")

## 5. VotingClassifier (Final Model)

In [ ]:
rf_best = rf_grid.best_estimator_
gb_best = gb_grid.best_estimator_

voting = VotingClassifier(
    estimators=[('rf', rf_best), ('gb', gb_best)],
    voting='soft'
)
voting.fit(X_train_s, y_train)

y_pred      = voting.predict(X_test_s)
y_pred_prob = voting.predict_proba(X_test_s)[:, 1]

print("=== VotingClassifier Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No Accident','Accident']))

## 6. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['No Acc','Acc'], yticklabels=['No Acc','Acc'],
            linewidths=1, cbar=False)
axes[0].set_title('Confusion Matrix', fontsize=13)
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#FF3B30', lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#FF3B30')
axes[1].set_title('ROC Curve', fontsize=13)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)

# Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_pred_prob)
axes[2].plot(rec, prec, color='#0A84FF', lw=2.5)
axes[2].fill_between(rec, prec, alpha=0.15, color='#0A84FF')
axes[2].set_title('Precision-Recall Curve', fontsize=13)
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance

In [ ]:
# Use RF component for feature importance
rf_component = rf_best
importances  = rf_component.feature_importances_
indices      = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(11, 5))
colors  = ['#FF3B30' if imp > 0.12 else '#FF9500' if imp > 0.07 else '#0A84FF'
           for imp in importances[indices]]
ax.bar(range(len(FEATURE_COLS)),
       importances[indices],
       color=colors, alpha=0.9, edgecolor='none')
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_xticklabels([FEATURE_COLS[i] for i in indices], rotation=35, ha='right', fontsize=10)
ax.set_title('Feature Importances (RandomForest Component)', fontsize=14, pad=12)
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cross-Validation

In [ ]:
cv_scores = cross_val_score(voting, X_train_s, y_train, cv=skf, scoring='f1')
print("5-Fold Cross-Validation F1 Scores:")
for i, s in enumerate(cv_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"  Mean:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 9. Save Artifacts

In [ ]:
os.makedirs('../backend/app/ml/artifacts', exist_ok=True)

joblib.dump(voting, '../backend/app/ml/artifacts/model.pkl')
joblib.dump(scaler, '../backend/app/ml/artifacts/scaler.pkl')

print("model.pkl  saved ✅")
print("scaler.pkl saved ✅")